# 1. setup

In [20]:
# imports

import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataset, DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments, LogitsProcessor
import torch.nn as nn
import numpy as np
import random
import os
import sys

sys.path.append("..")

from data_utils.scan_helpers import (
    load_scan_split,
    build_vocab,
)

from evaluation.scan_evaluation import (
    greedy_decode,
    sequence_accuracy,
    evaluate_sequence_accuracy,
    ordered_sequence_token_accuracy,   
    evaluate_ordered_token_accuracy,
)

from result_utils.save_results import (
    save_experiment_results,
    load_experiment_results,
)

from result_utils.plot_results import (
    exp1_plot_sequence_accuracy,
    exp2_plot_sequence_accuracy,
    exp2_plot_token_accuracy,
)

In [26]:
#Load pre-trained model and tokenizer

model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name, padding_side='left')
# Add special tokens to GPT-2 tokenizer
special_tokens = ["<SEP>", "<EOS>"]
tokenizer.add_tokens(special_tokens)
# Set pad token to avoid warnings (use EOS as pad)
tokenizer.pad_token = tokenizer.eos_token


model = GPT2LMHeadModel.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

Embedding(50259, 768)

In [22]:
# Adapt data for gpt-2 decoder-only architecture
def adapt_for_gpt2(sequence):
    """
    Convert SCAN sequence from 'IN: command OUT: actions' format to GPT-2 format.
    Format: 'command <SEP> actions <EOS>'
    """
    # Parse the sequence - format is "IN: input OUT: output"
    parts = sequence.split(" OUT: ")
    if len(parts) != 2:
        return None
    
    input_part = parts[0].replace("IN: ", "").strip()
    output_part = parts[1].strip()
    
    # Format for GPT-2: input <SEP> output <EOS>
    formatted = f"{input_part} <SEP> {output_part} <EOS>"
    return formatted

def create_gpt2_dataset_file(data_path, output_path):
    """
    Create a dataset file in the format GPT-2 can fine-tune on.
    """
    lines = []
    with open(data_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                formatted = adapt_for_gpt2(line)
                if formatted:
                    lines.append(formatted)
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, 'w') as f:
        for line in lines:
            f.write(line + "\n")
    
    return len(lines)

In [23]:
percentages = [1, 2, 4, 8, 16, 32, 64, 100]

dataset_pairs = []
path_pairs = {"length_split": [], "simple_split": []}
def create_and_save_adapted_datasets(type: str):
    if type == "simple_split":
        for percentage_cmds_used in percentages:
            if percentage_cmds_used == 100:
                train_path = "data/simple_split/tasks_train_simple.txt"
                test_path  = "data/simple_split/tasks_test_simple.txt"
            else:
                train_path = f"data/simple_split/size_variations/tasks_train_simple_p{percentage_cmds_used}.txt"
                test_path  = f"data/simple_split/size_variations/tasks_test_simple_p{percentage_cmds_used}.txt"

            #dataset_pairs.append((percentage_cmds_used, train_path, test_path))

            output_train_path = f"data/gpt2_finetune/simple_split/train_p{percentage_cmds_used}.txt"
            output_test_path  = f"data/gpt2_finetune/simple_split/test_p{percentage_cmds_used}.txt"

            create_gpt2_dataset_file(train_path, output_train_path)
            create_gpt2_dataset_file(test_path, output_test_path)

            path_pairs["simple_split"].append((percentage_cmds_used, output_train_path, output_test_path))

    elif type == "length_split":
        train_path = "data/length_split/tasks_train_length.txt"
        test_path  = "data/length_split/tasks_test_length.txt"

        output_train_path = f"data/gpt2_finetune/length_split/train_length.txt"
        output_test_path  = f"data/gpt2_finetune/length_split/test_length.txt"

        create_gpt2_dataset_file(train_path, output_train_path)
        create_gpt2_dataset_file(test_path, output_test_path)

        path_pairs["length_split"].append((output_train_path, output_test_path))

create_and_save_adapted_datasets("simple_split")
create_and_save_adapted_datasets("length_split")

In [ ]:
# Fine-tuning hyperparameters
SEED_BASE = 42
LEARNING_RATE = 7e-4
BATCH_SIZE = 8
EPOCHS = 3
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01
SAVE_STEPS = 1000
EVAL_STEPS = 1000
GRAD_CLIP = 1.0  # Gradient clipping
NUM_RUNS = 1 # for testing
#NUM_RUNS = 5  # Number of runs for statistical robustness

def tokenize_action_sequence(sequence_str):
    """Convert a space-separated action sequence string to a list of tokens."""
    return sequence_str.strip().split()

class OracleLengthLogitsProcessor(LogitsProcessor):
    """
    Logits processor that enforces oracle length constraints:
    - Forbids EOS before target_length
    - Forces EOS at target_length
    Mimics the oracle behavior from the original Transformer experiments.
    """
    def __init__(self, target_length, eos_token_id, start_length):
        super().__init__()
        self.target_length = target_length
        self.eos_token_id = eos_token_id
        self.start_length = start_length  # Length of input prompt
    
    def __call__(self, input_ids, scores):
        # Current generation step (number of new tokens generated, not including input)
        # input_ids includes both input and all previously generated tokens
        current_step = input_ids.shape[1] - self.start_length
        device = scores.device
        
        # Before target length: forbid EOS
        if current_step < self.target_length:
            scores[:, self.eos_token_id] = torch.tensor(float('-inf'), device=device)
        
        # At or after target length: force EOS
        elif current_step >= self.target_length:
            # Set all non-EOS tokens to very low probability
            scores[:] = torch.tensor(float('-inf'), device=device)
            scores[:, self.eos_token_id] = 0.0
        
        return scores

def compute_token_accuracy(predicted_tokens, target_tokens):
    """
    Compute position-wise token accuracy (partial credit).
    Matches the logic from ordered_sequence_token_accuracy in original experiments.
    NOTE: Both sequences should have EOS already removed before calling this.
    """
    if not target_tokens:
        return 0.0
    
    total_correct = 0
    total_tokens = 0
    
    # Compare position-wise
    L = min(len(predicted_tokens), len(target_tokens))
    for i in range(L):
        if predicted_tokens[i] == target_tokens[i]:
            total_correct += 1
        total_tokens += 1
    
    # If prediction is shorter than target, remaining tokens count as incorrect
    total_tokens += max(0, len(target_tokens) - L)
    
    return total_correct / total_tokens if total_tokens > 0 else 0.0

def generate_with_oracle(model, tokenizer, input_ids, target_length, eos_token_id):
    """
    Generate with oracle length constraints (matches oracle behavior from Transformer experiments).
    Forces EOS at target_length, forbids early EOS using LogitsProcessor.
    """
    from transformers import LogitsProcessorList
    
    # Create oracle logits processor
    oracle_processor = OracleLengthLogitsProcessor(
        target_length=target_length,
        eos_token_id=eos_token_id,
        start_length=input_ids.shape[1]
    )
    
    with torch.no_grad():
        generated = model.generate(
            input_ids,
            max_new_tokens=target_length + 2,  # Small buffer for generation
            num_beams=1,
            do_sample=False,
            logits_processor=LogitsProcessorList([oracle_processor]),
            pad_token_id=tokenizer.pad_token_id,
        )
    
    # Extract output after <SEP>
    generated_text = tokenizer.decode(generated[0], skip_special_tokens=False)
    if " <SEP> " in generated_text:
        output_part = generated_text.split(" <SEP> ")[1].replace(" <EOS>", "").strip()
    else:
        output_part = ""
    
    return output_part

def fine_tune_and_evaluate_gpt2(train_file, test_file, split_name, run_seed=None, 
                                  use_oracle=False, split_by_length=False, output_dir="gpt2_finetuned"):
    """
    Fine-tune and evaluate GPT-2 on a specific SCAN split.
    Evaluates using both sequence accuracy (exact match) and token accuracy (partial credit).
    
    Args:
        train_file: Path to training data file
        test_file: Path to test data file for evaluation
        split_name: Name of the split being trained on
        run_seed: Random seed for reproducibility (if None, uses SEED_BASE)
        use_oracle: If True, use oracle length constraints during generation
        split_by_length: If True, return results split by source/target length (for length split experiment)
        output_dir: Directory to save results
    
    Returns:
        Dictionary with evaluation results (sequence and token accuracy)
    """
    seed = run_seed if run_seed is not None else SEED_BASE
    
    oracle_str = "with oracle" if use_oracle else "without oracle"
    print(f"\n{'='*60}")
    print(f"Fine-tuning GPT-2 on {split_name} (seed={seed}, {oracle_str})")
    print(f"{'='*60}")
    
    # Set seeds for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    
    # Determine device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Create model and resize embeddings to include new tokens
    model = GPT2LMHeadModel.from_pretrained("gpt2")
    model.resize_token_embeddings(len(tokenizer))
    model.to(device)
    
    # Create datasets
    train_dataset = TextDataset(
        tokenizer=tokenizer,
        file_path=train_file,
        block_size=128,
    )
    
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=f"{output_dir}_{split_name}",
        overwrite_output_dir=True,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        save_steps=SAVE_STEPS,
        save_total_limit=1,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=GRAD_CLIP,
        logging_steps=10,
        seed=seed,
        disable_tqdm=False,  # Keep the built-in progress bar for lightweight progress updates
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=train_dataset,
    )
    
    # Train
    trainer.train()
    
    print(f"Fine-tuning on {split_name} completed!")
    print(f"\nEvaluating on {split_name} ({oracle_str})...")
    
    # Evaluate
    model.eval()
    
    if split_by_length:
        # For length split: organize by source and target length
        results_by_source_length = {}
        results_by_target_length = {}
    else:
        # For simple split: just overall metrics
        seq_correct = 0
        total_samples = 0
        total_token_acc = 0.0
    
    eos_token_id = tokenizer.convert_tokens_to_ids("<EOS>")
    
    # Load all test samples
    test_samples = []
    with open(test_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            # Parse input and expected output
            parts = line.split(" <SEP> ")
            if len(parts) != 2:
                continue
            
            input_text = parts[0]
            expected_output = parts[1].replace(" <EOS>", "").strip()
            
            # Calculate lengths
            input_tokens = tokenize_action_sequence(input_text)
            target_tokens = tokenize_action_sequence(expected_output)
            source_length = len(input_tokens)
            target_length = len(target_tokens)
            
            test_samples.append({
                "input_text": input_text,
                "expected_output": expected_output,
                "source_length": source_length,
                "target_length": target_length,
                "target_tokens": target_tokens,
            })
    
    # Batch generation
    batch_size = 32
    num_batches = (len(test_samples) + batch_size - 1) // batch_size
    
    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, len(test_samples))
        batch_samples = test_samples[start_idx:end_idx]
        
        # Prepare batch inputs
        input_texts = [s["input_text"] + " <SEP>" for s in batch_samples]
        encoded = tokenizer(input_texts, return_tensors="pt", padding=True, truncation=True)
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)
        
        # Generate outputs
        if use_oracle:
            # Oracle mode: process one by one (since target lengths differ)
            generated_outputs = []
            for i, sample in enumerate(batch_samples):
                single_input_ids = input_ids[i:i+1]
                gen_output = generate_with_oracle(
                    model, tokenizer, single_input_ids, sample["target_length"], eos_token_id
                )
                generated_outputs.append(gen_output)
        else:
            with torch.no_grad():
                output_ids = model.generate(
                    input_ids,
                    attention_mask=attention_mask,
                    max_length=input_ids.shape[1] + 50,
                    num_beams=1,
                    early_stopping=True,
                    eos_token_id=eos_token_id,
                    pad_token_id=tokenizer.pad_token_id,
                )
            
            # Decode batch outputs
            generated_outputs = []
            for i in range(len(batch_samples)):
                generated_text = tokenizer.decode(output_ids[i], skip_special_tokens=False)
                
                # Extract only the output part (after <SEP>)
                if " <SEP> " in generated_text:
                    generated_output = generated_text.split(" <SEP> ")[1].replace(" <EOS>", "").strip()
                else:
                    generated_output = ""
                generated_outputs.append(generated_output)
        
        # Process batch results
        for sample, generated_output in zip(batch_samples, generated_outputs):
            expected_output = sample["expected_output"]
            target_tokens = sample["target_tokens"]
            source_length = sample["source_length"]
            target_length = sample["target_length"]
            
            # Compute accuracies
            seq_match = 1.0 if generated_output == expected_output else 0.0
            predicted_tokens = tokenize_action_sequence(generated_output)
            token_acc = compute_token_accuracy(predicted_tokens, target_tokens)
            
            if split_by_length:
                # Store by source length
                if source_length not in results_by_source_length:
                    results_by_source_length[source_length] = {"seq": [], "tok": []}
                results_by_source_length[source_length]["seq"].append(seq_match)
                results_by_source_length[source_length]["tok"].append(token_acc)
                
                # Store by target length
                if target_length not in results_by_target_length:
                    results_by_target_length[target_length] = {"seq": [], "tok": []}
                results_by_target_length[target_length]["seq"].append(seq_match)
                results_by_target_length[target_length]["tok"].append(token_acc)
            else:
                seq_correct += seq_match
                total_token_acc += token_acc
                total_samples += 1
        
        if not split_by_length and (batch_idx + 1) % 10 == 0:
            print(f"  Processed {end_idx}/{len(test_samples)} samples...")
    
    if split_by_length:
        # Aggregate results by length
        result = {
            "source_length": {},
            "target_length": {},
        }
        
        for length, metrics in results_by_source_length.items():
            result["source_length"][length] = {
                "seq": float(np.mean(metrics["seq"])),
                "tok": float(np.mean(metrics["tok"])),
            }
        
        for length, metrics in results_by_target_length.items():
            result["target_length"][length] = {
                "seq": float(np.mean(metrics["seq"])),
                "tok": float(np.mean(metrics["tok"])),
            }
        
        print(f"Evaluated on {len(results_by_source_length)} source lengths, {len(results_by_target_length)} target lengths")
    else:
        seq_accuracy = seq_correct / total_samples if total_samples > 0 else 0
        avg_token_accuracy = total_token_acc / total_samples if total_samples > 0 else 0
        
        print(f"Sequence accuracy: {seq_accuracy*100:.2f}%")
        print(f"Token accuracy: {avg_token_accuracy*100:.2f}%")
        
        result = {
            "seq": seq_accuracy,
            "tok": avg_token_accuracy,
        }
    
    # Clean up model to save memory
    del model

    del trainer    
    
    return result

    #torch.cuda.empty_cache()    

In [25]:
# Fine-tune and evaluate on simple split (multiple runs for statistical robustness)
results_simple = {}

print(f"\n{'='*80}")
print(f"EXPERIMENT: Simple Split with varying training data sizes")
print(f"Running {NUM_RUNS} times per configuration for statistical robustness")
print(f"{'='*80}")

for percentage, train_data_simple, test_data_simple in path_pairs["simple_split"]:
    if percentage != 1: # for testing purposes, only run for 1%
        break
    print(f"\n{'='*60}")
    print(f"Processing: {percentage}% of commands")
    print(f"{'='*60}")
    
    run_results = []
    
    for run_idx in range(1, NUM_RUNS + 1):
        print(f"\n--- Run {run_idx}/{NUM_RUNS} ---")
        
        run_seed = SEED_BASE + run_idx
        result = fine_tune_and_evaluate_gpt2(
            train_file=train_data_simple,
            test_file=test_data_simple,
            split_name=f"simple_split_p{percentage}_run{run_idx}",
            run_seed=run_seed,
        )
        run_results.append(result)
    
    # Calculate statistics across runs
    seq_vals = np.array([r["seq"] for r in run_results])
    tok_vals = np.array([r["tok"] for r in run_results])
    
    results_simple[percentage] = {
        "seq": {
            "mean": float(seq_vals.mean()),
            "std": float(seq_vals.std(ddof=1)),
            "runs": seq_vals.tolist(),
        },
        "tok": {
            "mean": float(tok_vals.mean()),
            "std": float(tok_vals.std(ddof=1)),
            "runs": tok_vals.tolist(),
        },
    }
    
    print(f"\n{'='*60}")
    print(f"Results for {percentage}%:")
    print(f"  Sequence Acc: {seq_vals.mean()*100:.2f}% ± {seq_vals.std(ddof=1)*100:.2f}%")
    print(f"  Token Acc:    {tok_vals.mean()*100:.2f}% ± {tok_vals.std(ddof=1)*100:.2f}%")
    print(f"{'='*60}")

print(f"\n{'='*80}")
print(f"Simple Split Experiment Complete!")
print(f"{'='*80}")


EXPERIMENT: Simple Split with varying training data sizes
Running 5 times per configuration for statistical robustness

Processing: 1% of commands

--- Run 1/5 ---

Fine-tuning GPT-2 on simple_split_p1_run1 (seed=43, without oracle)


c:\Users\MatsEllingsen\OneDrive - University of Copenhagen\UCPH\fall2025\ANLP\GenSys\.venv\Lib\site-packages\transformers\data\datasets\language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
c:\Users\MatsEllingsen\OneDrive - University of Copenhagen\UCPH\fall2025\ANLP\GenSys\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,1.874400
20,1.480800
30,1.019300
40,0.722200
50,0.651600


Fine-tuning on simple_split_p1_run1 completed!

Evaluating on simple_split_p1_run1 (without oracle)...


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='le

...

  Processed 1600/20701 samples...


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='le

  Processed 1920/20701 samples...


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


KeyboardInterrupt: 

In [ ]:
# Fine-tune and evaluate on length split (with and without oracle)
train_data_length = path_pairs["length_split"][0][0]
test_data_length = path_pairs["length_split"][0][1]

print(f"\n{'='*80}")
print(f"EXPERIMENT: Length Split")
print(f"Running {NUM_RUNS} times for statistical robustness")
print(f"Evaluating both WITH and WITHOUT oracle")
print(f"{'='*80}")

results_length = []

for run_idx in range(1, NUM_RUNS + 1):
    print(f"\n{'='*70}")
    print(f"Run {run_idx}/{NUM_RUNS}")
    print(f"{'='*70}")
    
    run_seed = SEED_BASE + run_idx
    
    # Train once, then evaluate with both oracle modes
    print("\n--- Training model ---")
    
    # We need to train the model once and keep it for both evaluations
    # So we'll modify approach: train separately, then evaluate twice
    
    # For now, let's do two separate training runs (one per oracle mode)
    # This matches the original approach where evaluation is done after training
    
    run_result = {
        "source_length": {"without_oracle": {}, "with_oracle": {}},
        "target_length": {"without_oracle": {}, "with_oracle": {}},
    }
    
    # Evaluate WITHOUT oracle
    print("\n--- Evaluating WITHOUT oracle ---")
    result_no_oracle = fine_tune_and_evaluate_gpt2(
        train_file=train_data_length,
        test_file=test_data_length,
        split_name=f"length_split_run{run_idx}_no_oracle",
        run_seed=run_seed,
        use_oracle=False,
        split_by_length=True,
    )
    
    # Evaluate WITH oracle (need to retrain with same seed)
    print("\n--- Evaluating WITH oracle ---")
    result_with_oracle = fine_tune_and_evaluate_gpt2(
        train_file=train_data_length,
        test_file=test_data_length,
        split_name=f"length_split_run{run_idx}_with_oracle",
        run_seed=run_seed,
        use_oracle=True,
        split_by_length=True,
    )
    
    # Merge results
    run_result["source_length"]["without_oracle"] = result_no_oracle["source_length"]
    run_result["source_length"]["with_oracle"] = result_with_oracle["source_length"]
    run_result["target_length"]["without_oracle"] = result_no_oracle["target_length"]
    run_result["target_length"]["with_oracle"] = result_with_oracle["target_length"]
    
    results_length.append(run_result)

# Aggregate results across runs (matching experiment 2 format)
final_results_length = {
    "source_length": {"without_oracle": {}, "with_oracle": {}},
    "target_length": {"without_oracle": {}, "with_oracle": {}},
}

for split_type in ["source_length", "target_length"]:
    for oracle_flag in ["without_oracle", "with_oracle"]:
        # Get all lengths from first run
        lengths = results_length[0][split_type][oracle_flag].keys()
        
        for length in lengths:
            seq_vals = [
                run[split_type][oracle_flag][length]["seq"]
                for run in results_length
                if length in run[split_type][oracle_flag]
            ]
            tok_vals = [
                run[split_type][oracle_flag][length]["tok"]
                for run in results_length
                if length in run[split_type][oracle_flag]
            ]
            
            seq_vals = np.array(seq_vals)
            tok_vals = np.array(tok_vals)
            
            final_results_length[split_type][oracle_flag][length] = {
                "seq": {
                    "runs": seq_vals.tolist(),
                    "mean": float(seq_vals.mean()),
                    "std": float(seq_vals.std(ddof=1)),
                    "sem": float(seq_vals.std(ddof=1) / np.sqrt(NUM_RUNS)),
                },
                "tok": {
                    "runs": tok_vals.tolist(),
                    "mean": float(tok_vals.mean()),
                    "std": float(tok_vals.std(ddof=1)),
                    "sem": float(tok_vals.std(ddof=1) / np.sqrt(NUM_RUNS)),
                },
            }

print(f"\n{'='*80}")
print(f"Length Split Experiment Complete!")
print(f"{'='*80}")

# Results Summary

In [ ]:
# Display Simple Split Results
print("\n" + "="*80)
print("SIMPLE SPLIT RESULTS SUMMARY")
print("="*80)
print(f"{'% Commands':<15} {'Seq Acc (Mean)':<20} {'Token Acc (Mean)':<20}")
print("-"*80)

for percentage in sorted(results_simple.keys()):
    seq_mean = results_simple[percentage]["seq"]["mean"] * 100
    seq_std = results_simple[percentage]["seq"]["std"] * 100
    tok_mean = results_simple[percentage]["tok"]["mean"] * 100
    tok_std = results_simple[percentage]["tok"]["std"] * 100
    
    print(f"{percentage:<15} {seq_mean:6.2f}% ± {seq_std:5.2f}%    {tok_mean:6.2f}% ± {tok_std:5.2f}%")

print("\n" + "="*80)
print("LENGTH SPLIT RESULTS SUMMARY")
print("="*80)

# Display results by source length
for oracle_flag in ["without_oracle", "with_oracle"]:
    print(f"\n{oracle_flag.replace('_', ' ').upper()}:")
    print("-"*80)
    
    print("\nBy Source Length:")
    print(f"{'Length':<10} {'Seq Acc':<20} {'Token Acc':<20}")
    for length in sorted(final_results_length["source_length"][oracle_flag].keys()):
        seq_mean = final_results_length["source_length"][oracle_flag][length]["seq"]["mean"] * 100
        seq_std = final_results_length["source_length"][oracle_flag][length]["seq"]["std"] * 100
        tok_mean = final_results_length["source_length"][oracle_flag][length]["tok"]["mean"] * 100
        tok_std = final_results_length["source_length"][oracle_flag][length]["tok"]["std"] * 100
        print(f"{length:<10} {seq_mean:5.2f}% ± {seq_std:4.2f}%    {tok_mean:5.2f}% ± {tok_std:4.2f}%")
    
    print("\nBy Target Length:")
    print(f"{'Length':<10} {'Seq Acc':<20} {'Token Acc':<20}")
    for length in sorted(final_results_length["target_length"][oracle_flag].keys()):
        seq_mean = final_results_length["target_length"][oracle_flag][length]["seq"]["mean"] * 100
        seq_std = final_results_length["target_length"][oracle_flag][length]["seq"]["std"] * 100
        tok_mean = final_results_length["target_length"][oracle_flag][length]["tok"]["mean"] * 100
        tok_std = final_results_length["target_length"][oracle_flag][length]["tok"]["std"] * 100
        print(f"{length:<10} {seq_mean:5.2f}% ± {seq_std:4.2f}%    {tok_mean:5.2f}% ± {tok_std:4.2f}%")

print("\n" + "="*80)

In [ ]:
# Save results using the same format as experiments 1 & 2
save_experiment_results(
    results=results_simple,
    filename="gpt2_experiment_simple_split",
    experiment_nr="GPT2_Simple",
    description="GPT-2 Fine-tuning: Simple split with varying training data sizes",
)

save_experiment_results(
    results=final_results_length,
    filename="gpt2_experiment_length_split",
    experiment_nr="GPT2_Length",
    description="GPT-2 Fine-tuning: Length split with oracle and without oracle evaluation",
)

print("Results saved!")